# Finding artificial water bodies with Google Satellite Embeddings

Uses the hand-made pond polygons (`Cuerpos_Agua_AtarraIA_29Julio26`) as training
labels and the **AlphaEarth / Satellite Embedding V1** dataset (64-D per 10 m pixel)
to find similar water bodies in the Golfo de Morrosquillo zone.

**Why a classifier, not cosine similarity:** the embeddings are unit vectors, so
*everything* sits at cosine 0.8-0.9 and a single mean centroid cannot separate ponds
from the landscape. A Random Forest trained on pond vs. background learns the
discriminative boundary and reaches **~89% test accuracy** (see the eval cell).

Pipeline: load polygons -> 80/20 split -> sample pixels -> train RF -> evaluate on
held-out ponds -> classify the whole region -> export candidate polygons.

In [1]:
import ee, json, urllib.request

# One-time auth (uncomment if you have never authenticated on this machine):
# ee.Authenticate()

PROJECT = 'lofty-tokenizer-437115-e3'   # <-- your Cloud project
ee.Initialize(project=PROJECT)
print('Earth Engine ready:', ee.Number(1).add(1).getInfo() == 2)

Earth Engine ready: True


In [2]:
# ---- Inputs -------------------------------------------------------------
GEOJSON = 'Cuerpos_Agua_AtarraIA_29Julio26_wgs84.geojson'  # WGS84 polygons
YEAR = 2024                    # Satellite Embedding V1 covers 2017..2024
SEARCH = ee.Geometry.Rectangle([-75.74, 9.14, -75.36, 9.56])  # Golfo de Morrosquillo

gj = json.load(open(GEOJSON))
feats = [ee.Feature(ee.Geometry.Polygon(f['geometry']['coordinates']),
                    {'Id': f['properties']['Id']}) for f in gj['features']]
ponds = ee.FeatureCollection(feats)
print('pond polygons:', ponds.size().getInfo())

emb = (ee.ImageCollection('GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL')
       .filterDate(f'{YEAR}-01-01', f'{YEAR+1}-01-01')
       .filterBounds(SEARCH)
       .mosaic())
bands = emb.bandNames()
print('embedding bands:', bands.size().getInfo())

pond polygons: 1341


embedding bands: 64


## 1. Train / test split (80/20 by polygon)

Splitting by **polygon** (not by pixel) means every test pixel comes from a pond the
model never saw — an honest generalization estimate, no spatial leakage.

In [3]:
ponds_s = ponds.randomColumn('split', 42)
trainPonds = ponds_s.filter(ee.Filter.lt('split', 0.8))
testPonds  = ponds_s.filter(ee.Filter.gte('split', 0.8))
print('train polys:', trainPonds.size().getInfo(),
      ' test polys:', testPonds.size().getInfo())

def pond_pixels(fc, subsample, seed):
    p = emb.sampleRegions(collection=fc, scale=10, tileScale=8)
    return (p.randomColumn('r', seed).filter(ee.Filter.lt('r', subsample))
             .map(lambda f: f.set('class', 1)))

def land_pixels(n, seed):
    pts = ee.FeatureCollection.randomPoints(region=SEARCH, points=n, seed=seed)
    return emb.sampleRegions(collection=pts, scale=10, tileScale=8).map(
        lambda f: f.set('class', 0))

# positives from ponds, negatives from random background (ponds are ~0.25% of area,
# so random points almost never land inside one)
posTr = pond_pixels(trainPonds, 0.35, 1)
posTe = pond_pixels(testPonds, 1.0, 2)
negTr = land_pixels(9000, 101)
negTe = land_pixels(2000, 202)

trainSet = posTr.merge(negTr)
testSet  = posTe.merge(negTe)

train polys: 1072  test polys: 269


## 2. Train Random Forest and evaluate on the held-out test set

In [4]:
clf = ee.Classifier.smileRandomForest(numberOfTrees=300, minLeafPopulation=1) \
        .train(trainSet, 'class', bands)

tested = testSet.classify(clf)
cm = tested.errorMatrix('class', 'classification')
print('confusion matrix [actual 0/1 x pred 0/1]:', cm.getInfo())
print('overall accuracy:', round(cm.accuracy().getInfo(), 4))
print('kappa:', round(cm.kappa().getInfo(), 4))
pa = cm.producersAccuracy().getInfo(); ua = cm.consumersAccuracy().getInfo()
print('pond recall   :', round(pa[1][0], 3))
print('pond precision:', round(ua[0][1], 3))

confusion matrix [actual 0/1 x pred 0/1]: [[1845, 155], [849, 8042]]


overall accuracy: 0.9078


kappa: 0.729


pond recall   : 0.905
pond precision: 0.981


## 3. Final model + classify the whole region (probability)

Retrain on **all** ponds for the best map, output probability, and threshold.

In [5]:
posAll = pond_pixels(ponds, 0.35, 1)
negAll = land_pixels(9000, 101)
clfProb = (ee.Classifier.smileRandomForest(numberOfTrees=300, minLeafPopulation=1)
           .setOutputMode('PROBABILITY')
           .train(posAll.merge(negAll), 'class', bands))

prob = emb.classify(clfProb).rename('p').clip(SEARCH)

THRESH = 0.5   # raise toward 0.6-0.7 for fewer/cleaner detections
candidates = prob.gte(THRESH).selfMask()

# keep only NEW candidates (drop pixels already inside your polygons)
pondRaster = ee.Image().byte().paint(ponds, 1).unmask(0)
newCand = candidates.updateMask(pondRaster.Not())

## 4. Visual check (saves a PNG you can open)

In [6]:
bbox = ee.Geometry.Rectangle([-75.62, 9.24, -75.41, 9.45])  # the pond area
vis = prob.visualize(min=0, max=1,
        palette=['000004','3b0f70','8c2981','de4968','fe9f6d','fcfdbf'])
outline = ee.Image().byte().paint(ponds, 1, 1).visualize(palette=['00ffff'])
url = vis.blend(outline).getThumbURL(
        {'region': bbox, 'dimensions': 1100, 'format': 'png'})
urllib.request.urlretrieve(url, 'probability_map.png')
print('wrote probability_map.png  (bright = likely pond, cyan = your inputs)')

wrote probability_map.png  (bright = likely pond, cyan = your inputs)


## 5. Vectorize candidates and export to Drive as a shapefile

In [7]:
vectors = (newCand.rename('cand').reduceToVectors(
                geometry=SEARCH, scale=10, geometryType='polygon',
                eightConnected=True, maxPixels=1e13)
           .map(lambda f: f.set('area_m2', f.geometry().area(1)))
           .filter(ee.Filter.gte('area_m2', 400)))   # drop <4-pixel speckle

task = ee.batch.Export.table.toDrive(
    collection=vectors,
    description=f'candidate_ponds_Morrosquillo_{YEAR}',
    fileFormat='SHP')
task.start()
print('export started -> check https://code.earthengine.google.com Tasks, '
      'or your Google Drive. Task id:', task.id)

export started -> check https://code.earthengine.google.com Tasks, or your Google Drive. Task id: 7GHGDCA6ULYGVZPVVY4Q37MG
